# Indicadores del desarrollo mundial (Banco Mundial): panel de países

Los *World Development Indicators* del Banco Mundial reúnen más de 1.400 indicadores por país y año: población, PIB, salud, educación, empleo, energía, ambiente. Con la librería `wbgapi` se arma un **panel país × año** con los indicadores que usted elija, para Colombia, un grupo de países o todo el mundo.

|  |  |
|---|---|
| **Entidad** | Banco Mundial, World Development Indicators (WDI) |
| **Acceso** | API pública del Banco Mundial con la librería [`wbgapi`](https://pypi.org/project/wbgapi/) |
| **Una fila es** | un país en un año |
| **Cobertura** | ≈ 217 economías (sin agregados regionales) |
| **Periodo** | 1960 a 2024, según el indicador |
| **Tamaño** | con 13 indicadores, 217 países y 1990–2023: ≈ 7.400 filas |
| **Actualización** | varias veces al año |
| **Requisitos** | ninguno |
| **Tiempo de descarga** | menos de 1 minuto |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base



| Columna | Qué es | Tipo |
|---|---|---|
| `economy`, `pais` | Código ISO3 del país (p. ej. `COL`) y nombre | texto |
| `region`, `ingreso` | Región del Banco Mundial y grupo de ingreso | categórica |
| `anio` | Año | número |
| `esperanza_vida` | Esperanza de vida al nacer (años) · `SP.DYN.LE00.IN` | número |
| `pib_pc` | PIB per cápita en dólares constantes de 2015 · `NY.GDP.PCAP.KD` | número |
| `poblacion`, `urbano_pct` | Población total y % urbana | número |
| `fecundidad`, `mortalidad_infantil` | Hijos por mujer; muertes de menores de 5 años por mil nacidos | número |
| `matricula_primaria` | Tasa bruta de matrícula en primaria (%) | número |
| `desempleo` | Desempleo (% de la fuerza laboral, estimación OIT) | número |
| `acceso_electricidad`, `acceso_agua` | % de la población con electricidad y con agua básica | número |
| `bosque_pct`, `agro_pct_pib`, `gini` | % de tierra con bosque, agricultura en el PIB, índice de Gini | número |

Para buscar otros indicadores, escriba un tema en `BUSCAR` (en inglés) y copie el código al diccionario `INDICADORES`.

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Regresión | `esperanza_vida` | PIB per cápita, urbanización, acceso a agua y electricidad, fecundidad |
| Clasificación | `ingreso` (grupo de ingreso del país) | los indicadores sociales |
| Agrupamiento + PCA | tipologías de países en un año | indicadores escalados |
| Pronóstico | un indicador de Colombia | su propia serie anual |

**Cuidados antes de modelar**

- Hay **muchos faltantes** (el Gini, por ejemplo, no se mide todos los años). Revise el % faltante antes de elegir variables.
- El mismo país aparece en muchos años: valide **por país** (`GroupKFold`) para no aprender de memoria a cada país.
- Variables como PIB y población tienen colas largas: use logaritmos.
- `mortalidad_infantil` y `esperanza_vida` están muy correlacionadas: cuidado con usar una para predecir la otra.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl"), ("wbgapi", "wbgapi")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

`PAISES` acepta `"all"` (todos los países, sin agregados) o una lista de códigos ISO3, p. ej. `["COL", "PER", "ECU", "MEX"]`.

In [ ]:
PAISES    = "all"     # "all" = todos los países, o una lista ISO3: ["COL", "PER", "ECU", "CHL", "MEX"]
ANIO_INI  = 1990
ANIO_FIN  = 2023
INDICADORES = {       # código del Banco Mundial: nombre de la columna. Agregue o quite los que quiera
    "SP.DYN.LE00.IN":    "esperanza_vida",
    "NY.GDP.PCAP.KD":    "pib_pc",
    "SP.POP.TOTL":       "poblacion",
    "SP.URB.TOTL.IN.ZS": "urbano_pct",
    "SP.DYN.TFRT.IN":    "fecundidad",
    "SH.DYN.MORT":       "mortalidad_infantil",
    "SE.PRM.ENRR":       "matricula_primaria",
    "SL.UEM.TOTL.ZS":    "desempleo",
    "EG.ELC.ACCS.ZS":    "acceso_electricidad",
    "SH.H2O.BASW.ZS":    "acceso_agua",
    "AG.LND.FRST.ZS":    "bosque_pct",
    "NV.AGR.TOTL.ZS":    "agro_pct_pib",
    "SI.POV.GINI":       "gini",
}
BUSCAR = "poverty"    # palabra en inglés para buscar otros indicadores. None = no buscar

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. Buscar otros indicadores

La búsqueda devuelve el código y el nombre de cada indicador que contiene la palabra.

In [ ]:
import wbgapi as wb
if BUSCAR:
    hallados = pd.DataFrame([{"codigo": s["id"], "nombre": s["value"]} for s in wb.series.list(q=BUSCAR)])
    print(f"{len(hallados)} indicadores con '{BUSCAR}'")
    display(hallados.head(15))

## 5. Descarga

In [ ]:
crudo = wb.data.DataFrame(list(INDICADORES), PAISES, time=range(ANIO_INI, ANIO_FIN + 1),
                          skipAggs=True, numericTimeKeys=True, columns="series")
wbp = crudo.rename(columns=INDICADORES).reset_index().rename(columns={"time": "anio"})

info = pd.DataFrame([{"economy": e["id"], "pais": e["value"], "region": e["region"], "ingreso": e["incomeLevel"]}
                     for e in wb.economy.list(skipAggs=True)])
regiones = {r["code"]: r["name"] for r in wb.region.list()}
niveles = {i["id"]: i["value"] for i in wb.income.list()}
info["region"] = info["region"].map(regiones)
info["ingreso"] = info["ingreso"].map(niveles)
wbp = info.merge(wbp, on="economy", how="right").sort_values(["economy", "anio"]).reset_index(drop=True)
print(f"{entero(len(wbp))} filas: {wbp['economy'].nunique()} países × {wbp['anio'].nunique()} años")

## 6. Revisión de la base

In [ ]:
ficha_rapida(wbp)
wbp[wbp["economy"] == "COL"].tail()

*(Se completa al ejecutar.)*

In [ ]:
ultimo = wbp.dropna(subset=["esperanza_vida", "pib_pc"]).sort_values("anio").groupby("economy").tail(1)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.scatter(ultimo["pib_pc"], ultimo["esperanza_vida"], s=14, color=AZUL, alpha=0.7)
col = ultimo[ultimo["economy"] == "COL"]
ax.scatter(col["pib_pc"], col["esperanza_vida"], s=60, color=NARANJA, zorder=3)
for _, f in col.iterrows():
    ax.annotate("  Colombia", (f["pib_pc"], f["esperanza_vida"]), color=NARANJA)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(mtick.FuncFormatter(entero))
ax.set_xlabel("PIB per cápita (US$ de 2015, escala log)"); ax.set_ylabel("esperanza de vida (años)")
ax.set_title("Esperanza de vida frente a PIB per cápita (último año con dato)")
plt.tight_layout(); plt.show()
(wbp.drop(columns=["economy", "pais", "region", "ingreso", "anio"]).isna().mean() * 100).round(1).sort_values().rename("% faltante").to_frame().T

*(Se completa al ejecutar.)*

## 7. Guardar

In [ ]:
ruta = guardar(wbp, etiqueta("banco_mundial_panel", None if PAISES == "all" else "seleccion"))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/banco_mundial_panel.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.